In [1]:
# ============================================
# CORRECTED PLANT DISEASE TRAINING CODE
# ============================================

# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Paths
zip_path = "/content/drive/MyDrive/Plant Disease Detection Dataset.zip"
extract_path = "/content/plant_disease_dataset"

# Step 3: Unzip dataset
import zipfile
import os
import shutil

# Remove old extracted folder if exists
if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

# Step 4: Find correct folder that contains actual class folders
def find_actual_dataset_dir(root):
    current = root
    while True:
        subfolders = [f for f in os.listdir(current) if os.path.isdir(os.path.join(current, f))]
        print(f"\nChecking folder: {current}")
        print("Subfolders found:", subfolders[:10])

        # Stop if folder contains many class folders like Apple/Tomato/Potato/etc.
        if len(subfolders) > 2 and any(
            ("Apple" in f or "Tomato" in f or "Potato" in f or "Corn" in f or "Pepper" in f)
            for f in subfolders
        ):
            return current

        # If only one folder exists, go inside it
        if len(subfolders) == 1:
            current = os.path.join(current, subfolders[0])
        else:
            return current

dataset_dir = find_actual_dataset_dir(extract_path)

print("\nFinal dataset directory:", dataset_dir)
print("Actual class folders:")
print(os.listdir(dataset_dir))

Mounted at /content/drive
Dataset extracted successfully!

Checking folder: /content/plant_disease_dataset
Subfolders found: ['Plant Disease Detection Dataset']

Checking folder: /content/plant_disease_dataset/Plant Disease Detection Dataset
Subfolders found: ['Dataset']

Checking folder: /content/plant_disease_dataset/Plant Disease Detection Dataset/Dataset
Subfolders found: ['Tomato_healthy', 'Apple___Apple_scab', 'Tomato_Leaf_Mold', 'Tomato_Bacterial_spot', 'Apple___Black_rot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Tomato_Early_blight', 'Pepper__bell___healthy', 'Potato___healthy']

Final dataset directory: /content/plant_disease_dataset/Plant Disease Detection Dataset/Dataset
Actual class folders:
['Tomato_healthy', 'Apple___Apple_scab', 'Tomato_Leaf_Mold', 'Tomato_Bacterial_spot', 'Apple___Black_rot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Tomato_Early_blight', 'Pepper__bell___healthy', 

In [2]:
# Step 5: Import libraries
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import matplotlib.pyplot as plt
import numpy as np

# Step 6: Settings
img_size = (224, 224)
batch_size = 32
seed = 42

# Step 7: Load dataset correctly
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_dataset.class_names
num_classes = len(class_names)

print("\nClass names:")
for i, name in enumerate(class_names):
    print(f"{i}: {name}")
print("Number of classes:", num_classes)

Found 33701 files belonging to 22 classes.
Using 26961 files for training.
Found 33701 files belonging to 22 classes.
Using 6740 files for validation.

Class names:
0: Apple___Apple_scab
1: Apple___Black_rot
2: Apple___Cedar_apple_rust
3: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
4: Corn_(maize)___Common_rust_
5: Corn_(maize)___Northern_Leaf_Blight
6: Corn_(maize)___healthy
7: Pepper__bell___Bacterial_spot
8: Pepper__bell___healthy
9: Potato___Early_blight
10: Potato___Late_blight
11: Potato___healthy
12: Tomato_Bacterial_spot
13: Tomato_Early_blight
14: Tomato_Late_blight
15: Tomato_Leaf_Mold
16: Tomato_Septoria_leaf_spot
17: Tomato_Spider_mites_Two_spotted_spider_mite
18: Tomato__Target_Spot
19: Tomato__Tomato_YellowLeaf__Curl_Virus
20: Tomato__Tomato_mosaic_virus
21: Tomato_healthy
Number of classes: 22


In [3]:
# Step 8: Optimize performance
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)

# Step 9: Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
])

# Step 10: Base model
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)

# Step 11: Compile
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 22)             │        28,182 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,286,166 (8.72 MB)

 Trainable params: 28,182 (110.09 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [4]:
# Step 12: Initial training
epochs = 3

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=epochs
)

Epoch 1/3
843/843 ━━━━━━━━━━━━━━━━━━━━ 1795s 2s/step - accuracy: 0.7779 - loss: 0.7109 - val_accuracy: 0.8843 - val_loss: 0.3813
Epoch 2/3
843/843 ━━━━━━━━━━━━━━━━━━━━ 1809s 2s/step - accuracy: 0.8756 - loss: 0.3799 - val_accuracy: 0.9047 - val_loss: 0.3144
Epoch 3/3
843/843 ━━━━━━━━━━━━━━━━━━━━ 1788s 2s/step - accuracy: 0.8855 - loss: 0.3415 - val_accuracy: 0.9068 - val_loss: 0.2917


In [ ]:
# Step 13: Fine-tuning
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fine_tune_epochs = 5
total_epochs = epochs + fine_tune_epochs

history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1]
)

In [5]:
# Step 14: Save correct model
model_save_path = "/content/drive/MyDrive/plant_disease_mobilenetv2_fixed.keras"
model.save(model_save_path)
print("Saved model to:", model_save_path)

# Step 15: Save class names
class_file = "/content/drive/MyDrive/plant_disease_class_names_fixed.txt"
with open(class_file, "w") as f:
    for item in class_names:
        f.write(item + "\n")

print("Saved class names to:", class_file)

Saved model to: /content/drive/MyDrive/plant_disease_mobilenetv2_fixed.keras
Saved class names to: /content/drive/MyDrive/plant_disease_class_names_fixed.txt
